# Faster R-CNN + DLA - Colab Training (Self-contained)

DLA backbone + FPN'li Faster R-CNN. Tum moduller hucrelerde inline.
**Backbone egitim oncesi secilir** (dla46x_c .. dla169 araligi).

**Adimlar:**
1. GPU kontrolu
2. Drive mount + dataset path
3. Bagimlilik kurulum
4. Model factory (cfg['BACKBONE'] ile DLA varyanti secimi)
5. Dataset / metrics / Excel logger
6. Egitim konfig (BACKBONE secimi) + baslat


## 1. GPU kontrol

In [1]:
!nvidia-smi

Fri May  1 23:53:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Drive mount

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Bagimliliklar

Colab'da torch + torchvision zaten kurulu.

In [3]:
!pip install -q timm pycocotools albumentations openpyxl

## 4. Dataset yolu

COCO formatinda olmali:
```
<DATA_DIR>/
  train/  images/  _annotations.coco.json
  val/    images/  _annotations.coco.json
```

In [6]:
import os

DATA_DIR  = '/content/drive/MyDrive/Sayzek/dataset/dataset_augmented_04_23_2026/dataset_augmented'
RUNS_DIR  = '/content/drive/MyDrive/Sayzek/runs/DLA-34'

for split in ['train', 'val']:
    j = os.path.join(DATA_DIR, split, '_annotations.coco.json')
    i = os.path.join(DATA_DIR, split, 'images')
    print(f'{split}: ann={os.path.exists(j)}  imgs={os.path.exists(i)}')

os.makedirs(RUNS_DIR, exist_ok=True)

train: ann=True  imgs=True
val: ann=True  imgs=True


## 5. Model factory (DLA + FPN, varyant secilebilir)

`timm` DLA backbone (C2..C5 features) + torchvision FPN + LastLevelMaxPool. 5 FPN seviyesi (P2..P6).

**Desteklenen DLA varyantlari** (Faster R-CNN toplam param):

| Backbone     | Backbone | Total Faster R-CNN |
|--------------|----------|--------------------|
| `dla46x_c`   | 0.81M    | **17.81M** (en kucuk) |
| `dla46_c`    | 1.04M    | 18.05M             |
| `dla60x_c`   | 1.06M    | 18.07M             |
| `dla34`      | 15.23M   | 32.35M             |
| `dla60x`     | 16.33M   | 33.69M             |
| `dla60`      | 21.01M   | 38.37M             |
| `dla102x`    | 25.28M   | 42.65M             |
| `dla102`     | 32.24M   | 49.61M             |
| `dla102x2`   | 40.26M   | 57.62M             |
| `dla169`     | 52.36M   | 69.73M             |

Kucuk varyantlar (`_c` = compact) edge cihaz icin uygun, ancak feature kapasitesi dusuk.


In [7]:
from collections import OrderedDict
import torch.nn as nn
import timm

from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool


# Tum DLA varyantlarinin yaklasik buyuklukleri (Faster R-CNN toplam, num_classes=4).
# Egitim oncesi secim icin referans tablosu.
DLA_VARIANT_SIZES = {
    'dla46x_c':  {'backbone_M': 0.81,  'frcnn_total_M': 17.81},
    'dla46_c':   {'backbone_M': 1.04,  'frcnn_total_M': 18.05},
    'dla60x_c':  {'backbone_M': 1.06,  'frcnn_total_M': 18.07},
    'dla34':     {'backbone_M': 15.23, 'frcnn_total_M': 32.35},
    'dla60x':    {'backbone_M': 16.33, 'frcnn_total_M': 33.69},
    'dla60':     {'backbone_M': 21.01, 'frcnn_total_M': 38.37},
    'dla102x':   {'backbone_M': 25.28, 'frcnn_total_M': 42.65},
    'dla102':    {'backbone_M': 32.24, 'frcnn_total_M': 49.61},
    'dla102x2':  {'backbone_M': 40.26, 'frcnn_total_M': 57.62},
    'dla169':    {'backbone_M': 52.36, 'frcnn_total_M': 69.73},
}


def list_dla_variants() -> None:
    # Kullanim: list_dla_variants() — secim oncesi tablosu yazdirir.
    print(f"{'Backbone':12s}  {'Backbone':>10s}  {'Total Faster R-CNN':>20s}")
    print('-' * 48)
    for name, sz in DLA_VARIANT_SIZES.items():
        print(f"{name:12s}  {sz['backbone_M']:>9.2f}M  {sz['frcnn_total_M']:>19.2f}M")


class DLAFPN(nn.Module):
    # DLA (timm) + FPN on stages C2..C5 -> P2..P6.
    # `backbone_name` ile herhangi bir DLA varyanti secilebilir.
    def __init__(self, backbone_name: str = 'dla34', pretrained: bool = True, out_channels: int = 256):
        super().__init__()
        # out_indices=(2,3,4,5) -> reductions [4,8,16,32].
        self.body = timm.create_model(
            backbone_name,
            features_only=True,
            pretrained=pretrained,
            out_indices=(2, 3, 4, 5),
        )
        in_channels_list = self.body.feature_info.channels()
        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=out_channels,
            extra_blocks=LastLevelMaxPool(),
        )
        self.out_channels = out_channels
        self.backbone_name = backbone_name

    def forward(self, x):
        feats = self.body(x)
        x_dict = OrderedDict((str(i), f) for i, f in enumerate(feats))
        return self.fpn(x_dict)


def create_faster_rcnn_dla(
    num_classes:   int,
    backbone_name: str = 'dla34',
    pretrained:    bool = True,
) -> nn.Module:
    if backbone_name not in DLA_VARIANT_SIZES:
        raise ValueError(
            f"Unknown DLA variant: {backbone_name}. "
            f"Supported: {list(DLA_VARIANT_SIZES.keys())}"
        )
    backbone = DLAFPN(backbone_name=backbone_name, pretrained=pretrained, out_channels=256)
    anchor_sizes  = ((32,), (64,), (128,), (256,), (512,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    rpn_anchor    = AnchorGenerator(anchor_sizes, aspect_ratios)
    return FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=rpn_anchor,
    )


# Geriye uyumluluk: dla34'lu eski isim.
def create_faster_rcnn_dla34(num_classes: int, pretrained: bool = True) -> nn.Module:
    return create_faster_rcnn_dla(num_classes=num_classes, backbone_name='dla34', pretrained=pretrained)


def count_parameters(model: nn.Module) -> dict:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'total':              total,
        'trainable':          trainable,
        'total_millions':     total / 1e6,
        'trainable_millions': trainable / 1e6,
    }


# Mevcut varyantlari yazdir.
list_dla_variants()


## 6. Dataset (COCO -> Faster R-CNN target)

Tek `_annotations.coco.json` + `images/` klasoru. Target dict: boxes (pascal_voc), labels (1-based), image_id, area, iscrowd.

In [8]:
import json
from collections import defaultdict
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

import albumentations as A
from albumentations.pytorch import ToTensorV2


def build_train_transform() -> A.Compose:
    # Dataset zaten augmente edilmis, bu yuzden hafif tutuyoruz.
    return A.Compose(
        [
            A.HueSaturationValue(
                hue_shift_limit=int(0.015 * 180),
                sat_shift_limit=int(0.7 * 255),
                val_shift_limit=int(0.4 * 255),
                p=0.5,
            ),
            A.ToFloat(max_value=255.0),
            ToTensorV2(),
        ],
        bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.3),
    )


def build_eval_transform() -> A.Compose:
    return A.Compose(
        [
            A.ToFloat(max_value=255.0),
            ToTensorV2(),
        ],
        bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.0),
    )


class CocoDetectionDataset(Dataset):
    def __init__(self, images_dir: str, annotation_path: str, transform: Optional[A.Compose] = None):
        super().__init__()
        self.images_dir = Path(images_dir)
        self.transform  = transform

        with open(annotation_path, 'r') as f:
            coco = json.load(f)

        # Skip annotation entries whose image file is missing on disk.
        # Disk uzerinde dosyasi olmayan annotation girdilerini atla.
        import os
        try:
            disk_files = set(os.listdir(self.images_dir))
        except FileNotFoundError:
            disk_files = set()

        all_images = coco['images']
        self.images = [im for im in all_images if im['file_name'] in disk_files]
        dropped = len(all_images) - len(self.images)
        if dropped > 0:
            print(f'[CocoDataset] Skipped {dropped}/{len(all_images)} entries '
                  f'(missing files in {self.images_dir}).')

        kept_ids = {im['id'] for im in self.images}
        self.anns_by_image_id: dict = defaultdict(list)
        for ann in coco['annotations']:
            if ann['image_id'] in kept_ids:
                self.anns_by_image_id[ann['image_id']].append(ann)

        # Background label = 0; gercek siniflar 1'den baslar.
        sorted_cats = sorted(coco['categories'], key=lambda c: c['id'])
        self.cat_id_to_label: dict = {}
        self.class_names: list = []
        for i, cat in enumerate(sorted_cats, start=1):
            self.cat_id_to_label[cat['id']] = i
            self.class_names.append(cat['name'])
        self.num_classes = len(self.class_names) + 1

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        image_id = img_info['id']

        img_path = self.images_dir / img_info['file_name']
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f'Image not readable: {img_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_h, img_w = image.shape[:2]

        anns = self.anns_by_image_id.get(image_id, [])
        boxes, labels, areas, iscrowd = [], [], [], []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                continue
            x_min = max(0.0, float(x))
            y_min = max(0.0, float(y))
            x_max = min(float(img_w), float(x + w))
            y_max = min(float(img_h), float(y + h))
            if x_max <= x_min or y_max <= y_min:
                continue
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(self.cat_id_to_label[ann['category_id']])
            areas.append(float(ann.get('area', (x_max - x_min) * (y_max - y_min))))
            iscrowd.append(int(ann.get('iscrowd', 0)))

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, class_labels=labels)
            image  = transformed['image']
            boxes  = transformed['bboxes']
            labels = transformed['class_labels']
            areas   = [(b[2] - b[0]) * (b[3] - b[1]) for b in boxes]
            iscrowd = [0] * len(boxes)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        if len(boxes) == 0:
            boxes_t   = torch.zeros((0, 4), dtype=torch.float32)
            labels_t  = torch.zeros((0,),    dtype=torch.int64)
            areas_t   = torch.zeros((0,),    dtype=torch.float32)
            iscrowd_t = torch.zeros((0,),    dtype=torch.int64)
        else:
            boxes_t   = torch.as_tensor(boxes,   dtype=torch.float32)
            labels_t  = torch.as_tensor(labels,  dtype=torch.int64)
            areas_t   = torch.as_tensor(areas,   dtype=torch.float32)
            iscrowd_t = torch.as_tensor(iscrowd, dtype=torch.int64)

        target = {
            'boxes':    boxes_t,
            'labels':   labels_t,
            'image_id': torch.tensor([image_id], dtype=torch.int64),
            'area':     areas_t,
            'iscrowd':  iscrowd_t,
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

## 7. Custom precision/recall (YOLO tarzi)

Confidence threshold sweep, IoU>=0.5'te TP/FP/FN say, en yuksek F1'i veren threshold'da P/R/F1 dondur (overall + per-class).

In [9]:
from typing import Optional


def box_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])
    lt = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    rb = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    wh = np.clip(rb - lt, a_min=0, a_max=None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def count_tp_fp_fn_per_class(preds, targets, score_threshold, iou_threshold=0.5):
    tp, fp, fn = {}, {}, {}
    targets_by_id = {t['image_id']: t for t in targets}

    for pred in preds:
        target = targets_by_id.get(pred['image_id'])
        if target is None:
            mask = pred['scores'] >= score_threshold
            for lbl in pred['labels'][mask]:
                fp[int(lbl)] = fp.get(int(lbl), 0) + 1
            continue

        mask = pred['scores'] >= score_threshold
        p_boxes  = pred['boxes'][mask]
        p_labels = pred['labels'][mask]
        gt_boxes  = target['boxes']
        gt_labels = target['labels']

        gt_matched = np.zeros(len(gt_boxes), dtype=bool)

        if len(p_boxes) > 0 and len(gt_boxes) > 0:
            iou = box_iou_matrix(p_boxes, gt_boxes)
            scores_filtered = pred['scores'][mask]
            sort_idx = np.argsort(-scores_filtered)
            for i in sort_idx:
                pred_label = int(p_labels[i])
                same_label = (gt_labels == pred_label)
                candidates = same_label & (~gt_matched) & (iou[i] >= iou_threshold)
                if candidates.any():
                    j = int(np.argmax(np.where(candidates, iou[i], -1)))
                    tp[pred_label] = tp.get(pred_label, 0) + 1
                    gt_matched[j] = True
                else:
                    fp[pred_label] = fp.get(pred_label, 0) + 1
        else:
            for lbl in p_labels:
                fp[int(lbl)] = fp.get(int(lbl), 0) + 1

        for j, lbl in enumerate(gt_labels):
            if not gt_matched[j]:
                fn[int(lbl)] = fn.get(int(lbl), 0) + 1

    return {'tp': tp, 'fp': fp, 'fn': fn}


def _safe_pr_f1(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def compute_custom_pr(all_predictions, all_targets, class_names, iou_threshold=0.5, n_thresholds=50):
    thresholds = np.linspace(0.001, 0.999, n_thresholds)
    best_overall = {'precision': 0.0, 'recall': 0.0, 'f1': -1.0, 'threshold': 0.0}
    best_per_class = {name: {'precision': 0.0, 'recall': 0.0} for name in class_names}
    best_threshold_per_class_pr: Optional[dict] = None

    for thr in thresholds:
        counts = count_tp_fp_fn_per_class(all_predictions, all_targets, float(thr), iou_threshold)
        tp_d, fp_d, fn_d = counts['tp'], counts['fp'], counts['fn']
        total_tp = sum(tp_d.values())
        total_fp = sum(fp_d.values())
        total_fn = sum(fn_d.values())
        p, r, f1 = _safe_pr_f1(total_tp, total_fp, total_fn)
        if f1 > best_overall['f1']:
            best_overall = {'precision': p, 'recall': r, 'f1': f1, 'threshold': float(thr)}
            snapshot = {}
            for k, name in enumerate(class_names):
                cls_label = k + 1
                cp, cr, _ = _safe_pr_f1(
                    tp_d.get(cls_label, 0),
                    fp_d.get(cls_label, 0),
                    fn_d.get(cls_label, 0),
                )
                snapshot[name] = {'precision': cp, 'recall': cr}
            best_threshold_per_class_pr = snapshot

    if best_threshold_per_class_pr is not None:
        best_per_class = best_threshold_per_class_pr
    return best_overall, best_per_class

## 8. Excel logger (per-epoch metrik kaydi)

In [10]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

STATIC_HEADERS = [
    'Epoch', 'Learning Rate',
    'Loss Classifier (Train)', 'Loss Box Reg (Train)',
    'Loss Objectness (Train)', 'Loss RPN Box Reg (Train)',
    'Total Loss (Train)',
    'Loss Classifier (Val)', 'Loss Box Reg (Val)',
    'Loss Objectness (Val)', 'Loss RPN Box Reg (Val)',
    'Total Loss (Val)',
    'Precision (Val)', 'Recall (Val)', 'F1 (Val)',
    'mAP@0.5 (overall)', 'mAP@0.5:0.95 (overall)',
]

_THIN       = Side(border_style='thin', color='BBBBBB')
BORDER      = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
HEADER_FILL = PatternFill('solid', start_color='1F3864')
HEADER_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=10)
DATA_FONT   = Font(name='Arial', size=10)
ALT_FILL    = PatternFill('solid', start_color='EBF3FB')


def _build_headers(class_names):
    headers = list(STATIC_HEADERS)
    for name in class_names:
        headers.append(f'mAP@0.5 ({name})')
        headers.append(f'mAP@0.5:0.95 ({name})')
        headers.append(f'Precision ({name})')
        headers.append(f'Recall ({name})')
    return headers


def _apply_header(ws, class_names):
    headers = _build_headers(class_names)
    for ci, val in enumerate(headers, 1):
        c = ws.cell(row=1, column=ci, value=val)
        c.fill = HEADER_FILL
        c.font = HEADER_FONT
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = BORDER
    for i in range(1, len(headers) + 1):
        width = 10 if i <= len(STATIC_HEADERS) else 18
        ws.column_dimensions[get_column_letter(i)].width = width
    ws.row_dimensions[1].height = 40
    ws.freeze_panes = 'A2'


def _fmt(v):
    return None if v is None else round(float(v), 4)


def _sum_loss_components(losses):
    parts = [losses.get(k) for k in
             ('loss_classifier', 'loss_box_reg', 'loss_objectness', 'loss_rpn_box_reg')]
    valid = [v for v in parts if v is not None]
    return sum(valid) if valid else None


def build_excel(output_path, class_names):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Training Metrics'
    _apply_header(ws, class_names)

    cm = wb.create_sheet('Confusion Matrix')
    cm['A1'] = 'Confusion Matrix - Egitim sonrasi doldurulur.'
    cm['A1'].font = Font(name='Arial', bold=True, size=11)

    ch = wb.create_sheet('Charts')
    ch['A1'] = 'Grafikler bu sayfaya eklenecektir.'
    ch['A1'].font = Font(name='Arial', size=10, italic=True, color='555555')

    wb.save(output_path)


def append_epoch_row(output_path, epoch, lr, train_losses, val_losses,
                     val_metrics, per_class_metrics, class_names):
    wb = openpyxl.load_workbook(output_path)
    ws = wb['Training Metrics']
    data_row = ws.max_row + 1
    alt_fill = ALT_FILL if data_row % 2 == 0 else None

    train_total = _sum_loss_components(train_losses)
    val_total   = _sum_loss_components(val_losses)

    row = [
        epoch, round(lr, 8),
        _fmt(train_losses.get('loss_classifier')),
        _fmt(train_losses.get('loss_box_reg')),
        _fmt(train_losses.get('loss_objectness')),
        _fmt(train_losses.get('loss_rpn_box_reg')),
        _fmt(train_total),
        _fmt(val_losses.get('loss_classifier')),
        _fmt(val_losses.get('loss_box_reg')),
        _fmt(val_losses.get('loss_objectness')),
        _fmt(val_losses.get('loss_rpn_box_reg')),
        _fmt(val_total),
        _fmt(val_metrics.get('precision')),
        _fmt(val_metrics.get('recall')),
        _fmt(val_metrics.get('f1')),
        _fmt(val_metrics.get('map50')),
        _fmt(val_metrics.get('map50_95')),
    ]
    for name in class_names:
        m = per_class_metrics.get(name, {})
        row.append(_fmt(m.get('map50')))
        row.append(_fmt(m.get('map50_95')))
        row.append(_fmt(m.get('precision')))
        row.append(_fmt(m.get('recall')))

    for ci, val in enumerate(row, 1):
        c = ws.cell(row=data_row, column=ci, value=val)
        c.font = DATA_FONT
        c.alignment = Alignment(horizontal='center')
        c.border = BORDER
        if alt_fill:
            c.fill = alt_fill

    wb.save(output_path)

## 9. Train fonksiyonu

Manuel warmup + cosine LR, hybrid validation (BN-eval mode val loss + tam eval tahminler), pycocotools mAP, custom YOLO P/R/F1, Excel log, best/last checkpoint, early stopping, last.pt'den resume.

In [12]:
import math
import random
import time

from torch.utils.data import DataLoader
from torch.optim import SGD
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


DEFAULT_CFG = {
    'EPOCHS': 50, 'BATCH_SIZE': 8, 'PATIENCE': 10, 'WORKERS': 2, 'SEED': 0,
    'DEVICE': 'cuda:0',
    'LR0': 0.005, 'LRF': 0.01, 'MOMENTUM': 0.9, 'WEIGHT_DECAY': 0.0005,
    'WARMUP_EPOCHS': 2,
    'PRETRAINED': True,
    'PROJECT': RUNS_DIR,
    'NAME': 'faster_rcnn_dla34_colab',
    'OUTPUT_XLSX': 'training_metrics.xlsx',
    'LOG_INTERVAL': 50,
    'RESUME_FROM': None,
}


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_lr(epoch, total_epochs, warmup_epochs, lr0, lrf):
    if epoch < warmup_epochs:
        return lr0 * (epoch + 1) / max(1, warmup_epochs)
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return lr0 * (lrf + (1 - lrf) * cosine)


def set_bn_eval(model):
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
            m.eval()


def resolve_run_dir(project_dir, base_name):
    project = Path(project_dir)
    project.mkdir(parents=True, exist_ok=True)
    if not (project / base_name).exists():
        return project / base_name
    counter = 2
    while (project / f'{base_name}{counter}').exists():
        counter += 1
    return project / f'{base_name}{counter}'


def build_dataloaders(data_dir, cfg):
    data_dir = Path(data_dir)
    train_ds = CocoDetectionDataset(
        images_dir=str(data_dir / 'train' / 'images'),
        annotation_path=str(data_dir / 'train' / '_annotations.coco.json'),
        transform=build_train_transform(),
    )
    val_ds = CocoDetectionDataset(
        images_dir=str(data_dir / 'val' / 'images'),
        annotation_path=str(data_dir / 'val' / '_annotations.coco.json'),
        transform=build_eval_transform(),
    )
    train_loader = DataLoader(train_ds, batch_size=cfg['BATCH_SIZE'], shuffle=True,
                              num_workers=cfg['WORKERS'], collate_fn=collate_fn,
                              pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg['BATCH_SIZE'], shuffle=False,
                            num_workers=cfg['WORKERS'], collate_fn=collate_fn,
                            pin_memory=True)
    return train_loader, val_loader, train_ds.class_names, train_ds.num_classes


def train_one_epoch(model, optimizer, loader, device, epoch, log_interval=50):
    model.train()
    loss_sums = {'loss_classifier': 0.0, 'loss_box_reg': 0.0,
                 'loss_objectness': 0.0, 'loss_rpn_box_reg': 0.0}
    n_batches = 0
    epoch_start = time.time()

    for batch_idx, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for k, v in loss_dict.items():
            loss_sums[k] += v.item()
        n_batches += 1

        if (batch_idx + 1) % log_interval == 0:
            elapsed = time.time() - epoch_start
            avg_loss = sum(loss_sums.values()) / n_batches
            print(f'  [Epoch {epoch}] batch {batch_idx + 1}/{len(loader)}  '
                  f'avg_loss={avg_loss:.4f}  elapsed={elapsed:.1f}s')

    return {k: v / n_batches for k, v in loss_sums.items()}


def predictions_to_coco_format(predictions, image_ids):
    out = []
    for pred, img_id in zip(predictions, image_ids):
        boxes  = pred['boxes'].cpu().numpy()
        labels = pred['labels'].cpu().numpy()
        scores = pred['scores'].cpu().numpy()
        for box, label, score in zip(boxes, labels, scores):
            x_min, y_min, x_max, y_max = box
            out.append({
                'image_id':    int(img_id),
                'category_id': int(label) - 1,
                'bbox': [float(x_min), float(y_min),
                         float(x_max - x_min), float(y_max - y_min)],
                'score': float(score),
            })
    return out


def compute_per_class_map_coco(coco_eval, class_names):
    precision = coco_eval.eval['precision']
    per_class = {}
    for k, name in enumerate(class_names):
        p_50 = precision[0, :, k, 0, 2]
        p_50 = p_50[p_50 > -1]
        ap_50 = float(p_50.mean()) if len(p_50) > 0 else None
        p_all = precision[:, :, k, 0, 2]
        p_all = p_all[p_all > -1]
        ap_all = float(p_all.mean()) if len(p_all) > 0 else None
        per_class[name] = {'map50': ap_50, 'map50_95': ap_all}
    return per_class


def validate(model, loader, coco_gt, class_names, device):
    val_losses = {'loss_classifier': 0.0, 'loss_box_reg': 0.0,
                  'loss_objectness': 0.0, 'loss_rpn_box_reg': 0.0}
    n_batches = 0
    all_coco_results, all_predictions, all_targets = [], [], []
    val_start = time.time()

    for images, targets in loader:
        images_dev  = [img.to(device) for img in images]
        targets_dev = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Pass 1: val loss (train mode + BN frozen).
        model.train(); set_bn_eval(model)
        with torch.no_grad():
            loss_dict = model(images_dev, targets_dev)
        for k, v in loss_dict.items():
            val_losses[k] += v.item()
        n_batches += 1

        # Pass 2: predictions (full eval mode).
        model.eval()
        with torch.no_grad():
            predictions = model(images_dev)

        image_ids = [int(t['image_id'].item()) for t in targets]
        all_coco_results.extend(predictions_to_coco_format(predictions, image_ids))

        for pred, target in zip(predictions, targets):
            all_predictions.append({
                'image_id': int(target['image_id'].item()),
                'boxes':    pred['boxes'].cpu().numpy(),
                'labels':   pred['labels'].cpu().numpy(),
                'scores':   pred['scores'].cpu().numpy(),
            })
            all_targets.append({
                'image_id': int(target['image_id'].item()),
                'boxes':    target['boxes'].numpy(),
                'labels':   target['labels'].numpy(),
            })

    val_losses = {k: v / n_batches for k, v in val_losses.items()}

    map50, map50_95 = 0.0, 0.0
    per_class_map = {name: {'map50': None, 'map50_95': None} for name in class_names}

    if all_coco_results:
        coco_dt = coco_gt.loadRes(all_coco_results)
        coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
        coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()
        map50_95 = float(coco_eval.stats[0])
        map50    = float(coco_eval.stats[1])
        per_class_map = compute_per_class_map_coco(coco_eval, class_names)

    overall, per_class_pr = compute_custom_pr(all_predictions, all_targets,
                                              class_names, iou_threshold=0.5)

    per_class_combined = {}
    for name in class_names:
        m  = per_class_map.get(name, {'map50': None, 'map50_95': None})
        pr = per_class_pr.get(name, {'precision': None, 'recall': None})
        per_class_combined[name] = {**m, **pr}

    print(f'  Validation done in {time.time() - val_start:.1f}s')
    return {
        'losses': val_losses,
        'metrics': {
            'precision': overall['precision'], 'recall': overall['recall'],
            'f1': overall['f1'], 'map50': map50, 'map50_95': map50_95,
        },
        'per_class': per_class_combined,
    }


def load_resume_state(resume_path, model, optimizer, device):
    print(f'[FRCNN-DLA34] Resuming from {resume_path}')
    ckpt = torch.load(resume_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = int(ckpt['epoch']) + 1
    print(f'[FRCNN-DLA34] Resumed at epoch {start_epoch}')
    return {'start_epoch': start_epoch}


def load_best_state(best_path):
    if not best_path.exists():
        return {'best_map50': -1.0, 'best_epoch': -1}
    bckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    return {
        'best_map50': float(bckpt.get('metric', -1.0)),
        'best_epoch': int(bckpt.get('epoch', -1)),
    }


def train(data_dir, cfg=None):
    c = {**DEFAULT_CFG, **(cfg or {})}
    set_seed(c['SEED'])

    if c['DEVICE'].startswith('cuda') and not torch.cuda.is_available():
        print('[FRCNN-DLA34] CUDA not available, falling back to CPU.')
        c['DEVICE'] = 'cpu'

    is_resume = c.get('RESUME_FROM') is not None

    print(f"[FRCNN-DLA] Backbone:  {c['BACKBONE']} + FPN")
    print(f"[FRCNN-DLA34] Device:    {c['DEVICE']}")
    print(f"[FRCNN-DLA34] Batch:     {c['BATCH_SIZE']}, Epochs: {c['EPOCHS']}")
    print(f"[FRCNN-DLA34] LR0:       {c['LR0']}, LRF: {c['LRF']}")
    print(f"[FRCNN-DLA34] Patience:  {c['PATIENCE']}")

    train_loader, val_loader, class_names, num_classes = build_dataloaders(data_dir, c)
    print(f"[FRCNN-DLA34] Train: {len(train_loader.dataset)} images, {len(train_loader)} batches")
    print(f"[FRCNN-DLA34] Val:   {len(val_loader.dataset)} images, {len(val_loader)} batches")
    print(f"[FRCNN-DLA34] Classes: {class_names} (num_classes={num_classes} incl. bg)")

    model = create_faster_rcnn_dla(
        num_classes=num_classes,
        backbone_name=c['BACKBONE'],
        pretrained=c['PRETRAINED'],
    )
    model = model.to(c['DEVICE'])
    params = count_parameters(model)
    print(f"[FRCNN-DLA34] Total params: {params['total_millions']:.2f}M")

    optimizer = SGD([p for p in model.parameters() if p.requires_grad],
                    lr=c['LR0'], momentum=c['MOMENTUM'], weight_decay=c['WEIGHT_DECAY'])

    start_epoch, best_map50, best_epoch, patience_counter = 0, -1.0, -1, 0

    if is_resume:
        save_dir = Path(c['RESUME_FROM']).resolve().parent
        excel_path = save_dir / c['OUTPUT_XLSX']
        if not excel_path.exists():
            raise FileNotFoundError(f'Excel not found in resume dir: {excel_path}')
        print(f"[FRCNN-DLA34] Resume dir: {save_dir}")
        resume_state = load_resume_state(c['RESUME_FROM'], model, optimizer, c['DEVICE'])
        start_epoch = resume_state['start_epoch']
        best_state = load_best_state(save_dir / 'best.pt')
        best_map50 = best_state['best_map50']
        best_epoch = best_state['best_epoch']
        patience_counter = max(0, (start_epoch - 1) - best_epoch)
        print(f"[FRCNN-DLA34] Best so far: mAP@0.5={best_map50:.4f} @ epoch {best_epoch}")
    else:
        save_dir = resolve_run_dir(c['PROJECT'], c['NAME'])
        save_dir.mkdir(parents=True, exist_ok=True)
        excel_path = save_dir / c['OUTPUT_XLSX']
        print(f"[FRCNN-DLA34] Output dir: {save_dir}")
        build_excel(excel_path, class_names)

    val_gt_path = Path(data_dir) / 'val' / '_annotations.coco.json'
    coco_gt = COCO(str(val_gt_path))

    print(f"\n[FRCNN-DLA34] Starting training from epoch {start_epoch}...\n")

    for epoch in range(start_epoch, c['EPOCHS']):
        epoch_start = time.time()
        lr = compute_lr(epoch, c['EPOCHS'], c['WARMUP_EPOCHS'], c['LR0'], c['LRF'])
        for g in optimizer.param_groups:
            g['lr'] = lr

        print(f"=== Epoch {epoch}/{c['EPOCHS'] - 1}  lr={lr:.6f} ===")

        train_losses = train_one_epoch(model, optimizer, train_loader,
                                       c['DEVICE'], epoch, c['LOG_INTERVAL'])
        train_total = sum(train_losses.values())
        print(f"  Train losses: cls={train_losses['loss_classifier']:.4f}  "
              f"box={train_losses['loss_box_reg']:.4f}  "
              f"obj={train_losses['loss_objectness']:.4f}  "
              f"rpn={train_losses['loss_rpn_box_reg']:.4f}  total={train_total:.4f}")

        val_results = validate(model, val_loader, coco_gt, class_names, c['DEVICE'])
        val_losses, val_metrics, per_class = (
            val_results['losses'], val_results['metrics'], val_results['per_class']
        )
        val_total = sum(val_losses.values())
        print(f"  Val losses:   total={val_total:.4f}")
        print(f"  Val metrics:  P={val_metrics['precision']:.4f}  "
              f"R={val_metrics['recall']:.4f}  F1={val_metrics['f1']:.4f}  "
              f"mAP@0.5={val_metrics['map50']:.4f}  "
              f"mAP@0.5:0.95={val_metrics['map50_95']:.4f}")

        append_epoch_row(excel_path, epoch, lr, train_losses, val_losses,
                         val_metrics, per_class, class_names)

        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'cfg': c, 'class_names': class_names,
        }, save_dir / 'last.pt')

        if val_metrics['map50'] > best_map50:
            best_map50 = val_metrics['map50']
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'cfg': c, 'class_names': class_names,
                'metric': val_metrics['map50'],
            }, save_dir / 'best.pt')
            print(f"  >> NEW BEST mAP@0.5={best_map50:.4f} at epoch {best_epoch}")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{c['PATIENCE']})  "
                  f"best mAP@0.5={best_map50:.4f} @ epoch {best_epoch}")

        print(f"  Epoch time: {time.time() - epoch_start:.1f}s\n")

        if patience_counter >= c['PATIENCE']:
            print(f"[FRCNN-DLA34] Early stopping at epoch {epoch}")
            break

    print(f"\n[FRCNN-DLA34] Training finished.")
    print(f"  Best epoch:   {best_epoch}")
    print(f"  Best mAP@0.5: {best_map50:.4f}")
    print(f"  Output dir:   {save_dir}")

In [ ]:
import os, json

ann = json.load(open(f'{DATA_DIR}/train/_annotations.coco.json'))
img_dir = f'{DATA_DIR}/train/images'

disk_files = set(os.listdir(img_dir))
ann_files  = [im['file_name'] for im in ann['images']]

print('annotation count:', len(ann_files))
print('disk count:', len(disk_files))

missing = [f for f in ann_files if f not in disk_files][:10]
extra   = [f for f in disk_files if f not in set(ann_files)][:10]
print('missing on disk (first 10):', missing)
print('extra on disk (first 10):', extra)
print('sample disk file:', sorted(disk_files)[:3])
print('sample ann file:', ann_files[:3])


annotation count: 42602
disk count: 42602


## 10. Egitim konfig + baslat

**Onemli:** `BACKBONE` degerini egitim oncesi sec. Hucre 5'teki tablo referans.

Colab T4 (~15 GiB) icin BATCH=8 dla34'te rahat sigar. Dla60+ veya dla102+ icin BATCH=4 yap.
Local 4 GiB GPU varsa BATCH=2.


In [14]:
# DLA varyanti sec: dla46x_c (en kucuk, 17.81M) ... dla169 (en buyuk, 69.73M)
# Tablo icin: list_dla_variants() (hucre 5'te otomatik yazdirildi)
BACKBONE = 'dla34'   # <-- buradan degistir

DLA_CFG = {
    'EPOCHS':        2,
    'BATCH_SIZE':    8,
    'WORKERS':       2,
    'PATIENCE':      10,
    'LR0':           0.005,
    'LRF':           0.01,
    'MOMENTUM':      0.9,
    'WEIGHT_DECAY':  0.0005,
    'WARMUP_EPOCHS': 1,
    'BACKBONE':      BACKBONE,
    'PRETRAINED':    True,
    'DEVICE':        'cuda:0',
    'NAME':          f'faster_rcnn_{BACKBONE}_colab',
    'OUTPUT_XLSX':   'training_metrics.xlsx',
    'PROJECT':       RUNS_DIR,
    'LOG_INTERVAL':  50,
    'RESUME_FROM':   None,
}

print(f"Selected backbone: {BACKBONE}")
if BACKBONE in DLA_VARIANT_SIZES:
    sz = DLA_VARIANT_SIZES[BACKBONE]
    print(f"  Backbone params (approx):     {sz['backbone_M']:.2f}M")
    print(f"  Faster R-CNN total (approx):  {sz['frcnn_total_M']:.2f}M")

train(data_dir=DATA_DIR, cfg=DLA_CFG)


[FRCNN-DLA34] Backbone:  dla34 + FPN
[FRCNN-DLA34] Device:    cuda:0
[FRCNN-DLA34] Batch:     8, Epochs: 2
[FRCNN-DLA34] LR0:       0.005, LRF: 0.01
[FRCNN-DLA34] Patience:  10
[FRCNN-DLA34] Train: 42602 images, 5325 batches
[FRCNN-DLA34] Val:   1596 images, 200 batches
[FRCNN-DLA34] Classes: ['Person', 'Car', 'OtherVehicle'] (num_classes=4 incl. bg)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/63.1M [00:00<?, ?B/s]

[FRCNN-DLA34] Total params: 32.35M
[FRCNN-DLA34] Output dir: /content/drive/MyDrive/Sayzek/runs/DLA-34/faster_rcnn_dla34_colab
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!

[FRCNN-DLA34] Starting training from epoch 0...

=== Epoch 0/1  lr=0.005000 ===


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_14120/2594356909.py", line 75, in __getitem__
    raise FileNotFoundError(f'Image not readable: {img_path}')
FileNotFoundError: Image not readable: /content/drive/MyDrive/Sayzek/dataset/dataset_augmented_04_23_2026/dataset_augmented/train/images/thermal_v1_002294.jpg


## 11. (Opsiyonel) Resume from last.pt

In [ ]:
# RESUME_CFG = {**DLA_CFG, 'RESUME_FROM': os.path.join(RUNS_DIR, DLA_CFG['NAME'], 'last.pt')}
# train(data_dir=DATA_DIR, cfg=RESUME_CFG)
